In [7]:
import torch
import numpy as np
from utilities import KolmogorovFlowDataset
from utilities import SpectralConv2d, FNOBlock, FNOTimestepper, NavierStokesMFILoss, Trainer
from torch.utils.data import DataLoader, random_split


In [9]:
if __name__ == "__main__":

    # ── Configuración ──────────────────────────────────────
    DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
    H, W     = 64, 64
    BATCH    = 32
    EPOCHS   = 200
    LR       = 1e-3
    MODES    = 12       # modos de Fourier (recomendado: ≤ H//2 = 32)
    HIDDEN   = 64
    N_LAYERS = 4
    NU       = 1e-3     # viscosidad del flujo Kolmogorov
    DT       = 0.01     # paso de tiempo de la simulación
    ETA      = 1e-3     # penalización IBM

    print(f"Dispositivo: {DEVICE}")

    # ── Cargar dataset ─────────────────────────────────────
    # Descomenta y ajusta el path cuando tengas tu archivo .npz:
    #
    #   dataset = KolmogorovDataset(
    #       path     = "kolmogorov_data.npz",
    #       vort_key = "vorticity",   # (N, T, 64, 64)
    #       f_key    = "forcing",     # (64, 64)
    #       chi_key  = "chi",         # (64, 64)
    #   )
    #
    # Por ahora: dataset sintético para verificar que todo corre
    print("Generando dataset sintético (reemplaza con tus datos .npz)...")
    N_TRAJ, T = 50, 101
    w_fake   = np.random.randn(N_TRAJ, T, H, W).astype(np.float32)
    f_fake   = np.zeros((H, W), dtype=np.float32)
    # Forzamiento de Kolmogorov: f = sin(4y)
    y = np.linspace(0, 2 * np.pi, H, endpoint=False)
    f_fake[:] = np.sin(4 * y)[:, None]
    chi_fake = np.zeros((H, W), dtype=np.float32)
    np.savez("kolmogorov_synthetic.npz", vorticity=w_fake, forcing=f_fake, chi=chi_fake)

    dataset = KolmogorovFlowDataset("kolmogorov_synthetic.npz")

    # ── Split train / val ──────────────────────────────────
    n_val   = int(0.1 * len(dataset))
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val])

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  drop_last=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, drop_last=False, num_workers=0)
    print(f"Train: {n_train} muestras  |  Val: {n_val} muestras")

    # ── Modelo ─────────────────────────────────────────────
    model = FNOTimestepper(
        in_channels = 3,
        hidden_ch   = HIDDEN,
        modes1      = MODES,
        modes2      = MODES,
        n_layers    = N_LAYERS,
    ).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"Parámetros: {n_params:,}")

    # ── Pérdida physics-informed ────────────────────────────
    loss_fn = NavierStokesMFILoss(
        H=H, W=W,
        nu=NU, dt=DT, eta=ETA,
        alpha=1.0,   # peso pérdida datos
        beta=0.1,    # peso pérdida física (ajustar según convergencia)
        device=DEVICE,
    ).to(DEVICE)

    # ── Optimizador + scheduler ─────────────────────────────
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=10
    )

    # ── Entrenamiento ──────────────────────────────────────
    trainer = Trainer(model, loss_fn, optimizer, scheduler, DEVICE)
    history = trainer.fit(train_loader, val_loader, n_epochs=EPOCHS, log_every=20)

    # ── Rollout autoregresivo ──────────────────────────────
    print("\nGenerando trayectoria autoregresiva...")
    model.load_state_dict(torch.load("best_fno_ns_ibm.pt", map_location=DEVICE))

    # Condición inicial: primer frame de una trayectoria del val set
    w0_np   = val_ds.dataset.X[val_ds.indices[0], :1].unsqueeze(0)  # (1,1,H,W)
    f_t     = dataset.f_tensor                                        # (1,1,H,W)
    chi_t   = dataset.chi_tensor                                      # (1,1,H,W)

    traj = rollout(model, w0_np, f_t, chi_t, n_steps=100, device=DEVICE)
    print(f"Trayectoria generada: {traj.shape}")  # (1, 101, 64, 64)

    np.save("rollout_trajectory.npy", traj.numpy())
    print("Trayectoria guardada en rollout_trajectory.npy")

    # ── Guardar modelo final ───────────────────────────────
    torch.save({
        "model_state":  model.state_dict(),
        "config": {
            "in_channels": 3, "hidden_ch": HIDDEN,
            "modes1": MODES,  "modes2": MODES, "n_layers": N_LAYERS,
        },
        "norm": {"w_mean": dataset.w_mean, "w_std": dataset.w_std},
        "history": history,
    }, "fno_ns_ibm_final.pt")
    print("Modelo completo guardado en fno_ns_ibm_final.pt")

Dispositivo: cuda
Generando dataset sintético (reemplaza con tus datos .npz)...
Dataset cargado: 5000 pares  |  shape: torch.Size([5000, 3, 64, 64])
Train: 4500 muestras  |  Val: 500 muestras
Parámetros: 4,737,601
Época   20 | Train → total: 2.1104  data: 1.9837  phys: 1.2669 | Val   → total: 2.1157  data: 1.9843  phys: 1.3132


KeyboardInterrupt: 